## **Medicine Recommendation Model**
Algorithm: Naive Bayes Classifier
Data: 242 recommendations
Target: medicine_name (242 classes)

In [1]:
import pandas as pd
import numpy as np


In [32]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import pickle
import warnings
warnings.filterwarnings('ignore')


In [24]:
from google.colab import files
uploaded = files.upload()

Saving med_rec.csv to med_rec (1).csv


In [25]:
# Load data
recommendations = pd.read_csv('med_rec.csv')
print(f"Recommendations: {len(recommendations)}")
print(f"Unique medicines: {recommendations['medicine_name'].nunique()}")


Recommendations: 242
Unique medicines: 242


In [26]:
# Prepare features - Group by main categories instead of individual medicines
recommendations['text_features'] = recommendations['primary_conditions'].fillna('')



In [27]:
# Extract main category from primary_conditions
def extract_category(condition):
    if 'ANTI-INFECTIVE' in condition:
        return 'ANTI-INFECTIVE'
    elif 'CARDIOVASCULAR' in condition:
        return 'CARDIOVASCULAR'
    elif 'MENTAL' in condition or 'BEHAVIOURAL' in condition:
        return 'MENTAL_HEALTH'
    elif 'OPHTHALMOLOGICAL' in condition:
        return 'OPHTHALMOLOGICAL'
    elif 'ANTINEOPLASTIC' in condition or 'IMMUNOMODULATOR' in condition:
        return 'CANCER_TREATMENT'
    elif 'DIABETES' in condition or 'INSULIN' in condition:
        return 'DIABETES'
    elif 'RESPIRATORY' in condition or 'ASTHMA' in condition:
        return 'RESPIRATORY'
    elif 'GASTROINTESTINAL' in condition or 'STOMACH' in condition:
        return 'GASTROINTESTINAL'
    elif 'PAIN' in condition or 'ANALGESIC' in condition:
        return 'PAIN_MANAGEMENT'
    else:
        return 'OTHER'

recommendations['category'] = recommendations['primary_conditions'].apply(extract_category)
print("Category distribution:")
print(recommendations['category'].value_counts())


Category distribution:
category
OTHER               152
ANTI-INFECTIVE       62
MENTAL_HEALTH         8
CARDIOVASCULAR        8
OPHTHALMOLOGICAL      6
PAIN_MANAGEMENT       4
RESPIRATORY           1
GASTROINTESTINAL      1
Name: count, dtype: int64


In [28]:
# TF-IDF
tfidf_rec = TfidfVectorizer(max_features=500, stop_words='english')
X_text_rec = tfidf_rec.fit_transform(recommendations['text_features'])



In [29]:
# Label encoding
le_rec = LabelEncoder()
y_rec = le_rec.fit_transform(recommendations['category'])




In [30]:
# Train test split
X_train_rec, X_test_rec, y_train_rec, y_test_rec = train_test_split(X_text_rec, y_rec, test_size=0.2, random_state=42)

# Train model
rf_rec = RandomForestClassifier(n_estimators=100, random_state=42)
rf_rec.fit(X_train_rec, y_train_rec)


RandomForestClassifier(random_state=42)

In [37]:
# Evaluate
y_pred_rec = rf_rec.predict(X_test_rec)
print(f"Accuracy: {accuracy_score(y_test_rec, y_pred_rec):.3f}")
print("Classification Report:")
print(classification_report(y_test_rec, y_pred_rec, target_names=le_rec.classes_[np.unique(y_test_rec)]))

Accuracy: 0.959
Classification Report:
                  precision    recall  f1-score   support

  ANTI-INFECTIVE       1.00      1.00      1.00        12
  CARDIOVASCULAR       1.00      1.00      1.00         2
   MENTAL_HEALTH       1.00      1.00      1.00         1
OPHTHALMOLOGICAL       1.00      0.50      0.67         2
           OTHER       0.94      1.00      0.97        30
 PAIN_MANAGEMENT       1.00      0.50      0.67         2

        accuracy                           0.96        49
       macro avg       0.99      0.83      0.88        49
    weighted avg       0.96      0.96      0.95        49



In [39]:
# Save model
pickle.dump(rf_rec, open('recommendation_model.pkl', 'wb'))
pickle.dump(tfidf_rec, open('recommendation_tfidf.pkl', 'wb'))
pickle.dump(le_rec, open('recommendation_label_encoder.pkl', 'wb'))
print("Model saved!")


Model saved!


In [40]:
from google.colab import files
files.download('recommendation_model.pkl')
files.download('recommendation_tfidf.pkl')
files.download('recommendation_label_encoder.pkl')
print("Medicine Recommendation model downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Medicine Recommendation model downloaded!
